# CCGViewer example

In [12]:
!python ./run_ccg_gui.py

/Users/selinl/miniforge3/envs/NeuroPy2/lib/python3.11/site-packages/xarray/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Error importing CuPy
[open_project] 'test2' scans its own source; ignoring the 16 session(s) passed in
[CCGPointer] loaded 64 pointers ← /Users/selinl/Documents/ms_synchrony/NeuroPy/data/project_test2/ccg/pointers/test2_lowres_*.hkl
QGridLayoutEngine::addItem: Cell (2, 2) already taken
[CCGDataset] lowres loaded (1/1 session(s))
[CCGPointer] loaded 64 pointers ← /Users/selinl/Documents/ms_synchrony/NeuroPy/data/project_test2/ccg/pointers/test2_lowres_*.hkl
[CCGPanel] RENDER FAILED: no pairs in sel_data (nav.key=sess_RatN_Day2.ex_E.type_pyr-inter) — plot needs at least one pair in lists
[CCGReviewUI] ccg_ui=/Users/selinl/Documents/ms_sync

In [129]:
import numpy as np
position = np.load('/Users/selinl/Downloads/RatJ_Day2_2019-06-02_03-59-19_position.npy',allow_pickle=True)

In [101]:
import neuropy.io.optitrackio as optitrackio
x=optitrackio.posfromFBX('/Users/selinl/Downloads/optitrack/Take 2019-05-31 03.55.25 AM.fbx')

In [104]:
import numpy as np
np.array(x).shape[1]

6939617

In [109]:
6939617/125

55516.936

In [110]:
import subjects
subjects.sd.allsess[0].neurons.spiketrains[0]

ProcessData(RatJ_Day1_2019-05-31_03-55-36.xml)

In [89]:
6939617/56092 (120Hz)

123.71848035370463

In [91]:
y=optitrackio.getnframes_fbx('/Users/selinl/Downloads/Take 2019-05-31 03.55.25 AM.fbx')

In [86]:
56092/3600

15.581111111111111

In [63]:
# manually filling missing paradigm labels
import shutil
for s in subjects.sd.allsess:
    print('before',s.paradigm)
    if s.paradigm.label_slice('sd').n_epochs == 0:
        s.paradigm=s.paradigm.add_epoch_manually(label="sd", 
                            start=s.paradigm.label_slice('maze').stops[0], 
                            stop=s.paradigm.label_slice('maze').stops[0]+3600*5,
                            merge_dt=None)
    if s.paradigm.label_slice('rs').n_epochs == 0:
        s.paradigm=s.paradigm.add_epoch_manually(label="rs", 
                            start=s.paradigm.label_slice('sd').stops[0], 
                            stop=s.paradigm.label_slice('post').stops[0],
                            merge_dt=None)
    print('after',s.paradigm)
    src = s.filePrefix.with_suffix(".paradigm.npy")
    shutil.copy2(src, src.with_suffix(".orig.npy"))
    s.paradigm.save(src)

before 4 epochs
Snippet: 
    start   stop label
0      0  11160   pre
1  11185  15021  maze
2  15146  56092  post
3  15146  33146    sd
after 5 epochs
Snippet: 
    start   stop label
0      0  11160   pre
1  11185  15021  maze
2  15146  56092  post
3  15146  33146    sd
4  33146  56092    rs
/Users/selinl/Documents/ms_synchrony/bapun/RatJ/Day1/RatJ_Day1_2019-05-31_03-55-36.paradigm.npy saved
before 3 epochs
Snippet: 
    start   stop label
0      0  12106   pre
1  12408  15385  maze
2  15417  51619  post
after 5 epochs
Snippet: 
    start   stop label
0      0  12106   pre
1  12408  15385  maze
2  15385  33385    sd
3  15417  51619  post
4  33385  51619    rs
/Users/selinl/Documents/ms_synchrony/bapun/RatK/Day1/RatK_Day1_2019-08-06_03-44-01.paradigm.npy saved
before 4 epochs
Snippet: 
    start   stop  duration label
0      0  11280   11280.0   pre
1  11476  14760    3284.0  maze
2  15120  54600   39480.0  post
3  15120  33120       NaN    sd
after 5 epochs
Snippet: 
    start   stop

In [2]:
import neuropy.analyses.ms_connectivity as msconn

In [3]:
from neuropy.analyses.ms_connectivity import open_project
nd2, cd2, sd2 = open_project('test2')

[CCGPointer] loaded 64 pointers ← /Users/selinl/Documents/ms_synchrony/NeuroPy/data/project_test2/ccg/pointers/test2_lowres_*.hkl
[CCGDataset] lowres loaded (16/16 session(s))


In [3]:
k = msconn.Key(session='RatU_Day2NSD', resolution='lowres')
it = cd2.ccg_for(k)
print(it)

CCGData(session=RatU_Day2NSD, resolution=lowres, shape=(1, 188, 188, 21))


In [4]:
post1 = cd2.ccg_for(k.change(segment='post_half1'))
post2 = cd2.ccg_for(k.change(segment='post_half2'))

In [5]:
sd2.ensure_groups_loaded_for([k.session])
inds = sd2.groups.pairs_in_group('best', k.session)
ptr_key = k.add(conn_type=('pyr','inter'),excitability='E')
inds = sd2.groups.pairs_in_group('best', k.session) & cd2.ptr[ptr_key].pair_set

In [6]:
import numpy as np
arr=np.array(sorted(inds)) 
arr.shape

(48, 2)

In [7]:
def lag_window_bins(start: float, end: float, bin_size: float,
                    center_bin: int) -> tuple[int, int]:
    """Half-open bins [lo, hi) whose spans intersect lags *start*..*end* (seconds).

    Bins are centred on k*bin_size, so bin k spans (k-0.5)..(k+0.5)*bin_size."""
    if end < start:
        raise ValueError(f"window end {end} precedes start {start}")
    lo = center_bin + int(np.floor(start / bin_size + 0.5 + 1e-9))
    hi = center_bin + int(np.ceil(end / bin_size - 0.5 - 1e-9)) + 1
    return lo, max(hi, lo + 1)

start,end = lag_window_bins(1,3,1,10)

In [8]:
i1, i2 = cd2.segment_index(k, 'post1'), cd2.segment_index(k, 'post2')

In [33]:
v1= post1.pval_corrected[i1, arr[:,0], arr[:,1], start:end].min(-1)#.clip(0.01,1)
v1ns = v1>.05

In [34]:
v2=post2.pval_corrected[i2, arr[:,0], arr[:,1], start:end].min(-1)#.clip(0.01,1)
v2ns = v2>.05

In [35]:
arr[(v1ns==True) & (v2ns==False)]

array([], shape=(0, 2), dtype=int64)

In [36]:
arr[(v1ns==False) & (v2ns==True)]

array([], shape=(0, 2), dtype=int64)

In [37]:
v1

array([-1.19754653e-18,  1.26011609e-09, -4.38112677e-32, -0.00000000e+00,
        5.48323335e-15,  4.05950538e-15, -0.00000000e+00, -1.63821600e-39,
        0.00000000e+00, -3.94609768e-31, -0.00000000e+00, -0.00000000e+00,
       -0.00000000e+00, -0.00000000e+00, -3.25705414e-38,  0.00000000e+00,
       -0.00000000e+00, -4.79791087e-22, -0.00000000e+00,  0.00000000e+00,
       -0.00000000e+00, -0.00000000e+00, -0.00000000e+00, -0.00000000e+00,
       -0.00000000e+00, -0.00000000e+00, -0.00000000e+00, -2.70257616e-35,
       -0.00000000e+00, -0.00000000e+00,  0.00000000e+00, -0.00000000e+00,
       -0.00000000e+00, -5.78528285e-33, -1.11739635e-19, -2.86400778e-19,
       -1.65788931e-32, -0.00000000e+00, -0.00000000e+00, -0.00000000e+00,
       -7.65906455e-25, -0.00000000e+00, -1.03184817e-18, -2.53503223e-27,
       -0.00000000e+00, -0.00000000e+00,  0.00000000e+00, -0.00000000e+00])

In [38]:
v2

array([-1.19754653e-18,  1.26011609e-09, -4.38112677e-32, -0.00000000e+00,
        5.48323335e-15,  4.05950538e-15, -0.00000000e+00, -1.63821600e-39,
        0.00000000e+00, -3.94609768e-31, -0.00000000e+00, -0.00000000e+00,
       -0.00000000e+00, -0.00000000e+00, -3.25705414e-38,  0.00000000e+00,
       -0.00000000e+00, -4.79791087e-22, -0.00000000e+00,  0.00000000e+00,
       -0.00000000e+00, -0.00000000e+00, -0.00000000e+00, -0.00000000e+00,
       -0.00000000e+00, -0.00000000e+00, -0.00000000e+00, -2.70257616e-35,
       -0.00000000e+00, -0.00000000e+00,  0.00000000e+00, -0.00000000e+00,
       -0.00000000e+00, -5.78528285e-33, -1.11739635e-19, -2.86400778e-19,
       -1.65788931e-32, -0.00000000e+00, -0.00000000e+00, -0.00000000e+00,
       -7.65906455e-25, -0.00000000e+00, -1.03184817e-18, -2.53503223e-27,
       -0.00000000e+00, -0.00000000e+00,  0.00000000e+00, -0.00000000e+00])

In [15]:
np.divide(v1, v2, out=np.zeros_like(v1), where=v2!=0)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [31]:
from neuropy.analyses.ccg_transforms import ConnStrengthConfig, NormalizeBy
active_norms = {NormalizeBy.TOTAL_AREA}
for lo,hi in [(16,21),(0,8),(11,14)]:
    ptr_key1 = ptr_key.add(segment='post_half1')
    cfg = ConnStrengthConfig('conv', 'CS',
                            lo, hi)
    grid = cd2.get_conn_strength_for(ptr_key1, active_norms, cfg)
    vals = [grid[r, t] for r, t in arr]
    print(max(vals), min(vals), np.mean(vals), np.median(vals))

0.036727879799666116 0.0007912087912087945 0.008104984521564197 0.0052894264712525715
0.05311778290993072 0.004219780219780228 0.013233445089195555 0.01104938109264324
0.25192193968066234 0.017749652724185827 0.059252267272969554 0.04517342538866585


In [32]:
ptr_key2 = ptr_key.add(segment='post_half2')
for lo,hi in [(16,21),(0,8),(11,14)]:
    cfg = ConnStrengthConfig('conv', 'CS',
                            lo, hi)
    grid = cd2.get_conn_strength_for(ptr_key2, active_norms, cfg)
    vals = [grid[r, t] for r, t in arr]
    print(max(vals), min(vals), np.mean(vals), np.median(vals))

0.02231237322515213 0.00011147029316686818 0.006515129083357063 0.005373160236071078
0.04462474645030426 0.0016588463908242049 0.00979471467935111 0.008805743267730298
0.30773295754800073 0.014054627419782549 0.06918384512630994 0.05599953018557669


In [18]:
vals = cd2.source_config(k, 'post_half1').firing_rates
print(max(vals), min(vals), np.mean(vals), np.median(vals))
vals = cd2.source_config(k, 'post_half2').firing_rates
print(max(vals), min(vals), np.mean(vals), np.median(vals))

37.27855822550832 0.0 2.203996080282115 0.5411891558841652
35.32803450400493 0.00018484288354898336 2.298951246050786 0.5222119531731362


In [27]:
src1 = cd2._ccg[k].sources['post_half1']
src2 = cd2._ccg[k].sources['post_half2']
nspikes1=cd2.nd.sliced_neurons_for(src1)[0].n_spikes
print(max(nspikes1), min(nspikes1), np.mean(nspikes1), np.median(nspikes1))
nspikes2=cd2.nd.sliced_neurons_for(src2)[0].n_spikes
print(max(nspikes2), min(nspikes2), np.mean(nspikes2), np.median(nspikes2))

[dbg:_theme_intervals] sess=RatU_Day2NSD theme='paradigm' whitelist=['post'] raw_lbls=['maze', 'post', 'pre', 're-maze'] t0=12856.0 t1=29086.0 ep=yes
[dbg:_theme_intervals] EpochFilter→ type=tuple val=([(12856.0, 29086.0)], 16230.0)
605031 0 35770.85638297872 8783.5
[dbg:_theme_intervals] sess=RatU_Day2NSD theme='paradigm' whitelist=['post'] raw_lbls=['maze', 'post', 'pre', 're-maze'] t0=29086.0 t1=45316.0 ep=yes
[dbg:_theme_intervals] EpochFilter→ type=tuple val=([(29086.0, 45316.0)], 16230.0)
573374 3 37311.97872340425 8475.5


In [ ]:
lo,hi=11,14
cs_vals=[]
for seg_i in range(4): 
    ptr_key_i = ptr_key.add(segment=f'post_quarter{seg_i+1}')
    cfg = ConnStrengthConfig('conv', 'CS', lo, hi)
    grid = cd2.get_conn_strength_for(ptr_key_i, active_norms, cfg)
    vals = [grid[r, t] for r, t in arr]
    cs_vals.append(vals)
np.array(cs_vals)

array([[0.03277712, 0.0336    , 0.13535912, 0.03690037, 0.03524229,
        0.02698413, 0.03439555, 0.01851418, 0.13358169, 0.07498595,
        0.08978145, 0.06725664, 0.04181185, 0.08207934, 0.03444004,
        0.02671793, 0.0649534 , 0.03255814, 0.04022989, 0.05787435,
        0.01428571, 0.03352273, 0.03454231, 0.05193992, 0.06454439,
        0.03056546, 0.06619505, 0.04160166, 0.03585888, 0.04036827,
        0.18958611, 0.04259635, 0.06675749, 0.07250396, 0.02694136,
        0.01767352, 0.04676826, 0.21601686, 0.06081081, 0.02633033,
        0.05413687, 0.034631  , 0.05639396, 0.02777022, 0.0493558 ,
        0.0209286 , 0.09840783, 0.02856069],
       [0.04465075, 0.03753609, 0.14107884, 0.02773792, 0.07526882,
        0.02237136, 0.03774283, 0.01952515, 0.12850183, 0.07729071,
        0.09217877, 0.08673469, 0.04781627, 0.0910899 , 0.02092199,
        0.024828  , 0.08480229, 0.02937674, 0.08880309, 0.05701754,
        0.11029412, 0.0353238 , 0.03262274, 0.04045802, 0.07042572,
   

In [ ]:
for i in range(4):
    print(f'post_quarter{i+1}: {np.mean(cs_vals[i])}')

In [26]:
import numpy as np
label_firing_all = {}
label_ratio_all = {}
label_firing_all_values = {}
label_ratio_all_values = {}
for key in nd2.session_keys:
    print(key)
    neu=nd2.neurons_for(key)
    themes=nd2.get_themes(key)['paradigm']
    neu_slice = neu.behav_slice(behav_times=themes,labels='maze')
    neu_nonmaze = neu.behav_slice(behav_times=themes,labels=['pre','post'])
    inds=np.argsort(neu_slice.firing_rate)[::-1]
    label_highfiring = inds[(neu_slice.firing_rate>0.5) & (neu_slice.neuron_type=='pyr')]
    label_lowfiring = inds[(neu_slice.firing_rate<=0.5) & (neu_slice.neuron_type=='pyr')]
    label_ratio = neu_nonmaze.firing_rate/neu_slice.firing_rate
    label_firing_all[key] = np.full(len(neu_slice.firing_rate), None, dtype=object)
    label_firing_all[key][label_highfiring] = 'high'
    label_firing_all[key][label_lowfiring] = 'low'
    label_ratio_all[key] = label_ratio > 1
    label_firing_all_values[key] = neu_slice.firing_rate
    label_ratio_all_values[key] = label_ratio
    # for x in inds:
    #     print(f"{x}: {neu_slice.firing_rate[x]:.2f} {neu.firing_rate[x]} {neu_slice.neuron_type[x]}")
print(label_firing_all,label_ratio_all)

sess_RatJ_Day2
sess_RatK_Day2
sess_RatN_Day2


/var/folders/59/t44jf7m12zb1989pvvf2p6bm0000gn/T/ipykernel_1458/3303140680.py:15: RuntimeWarning: divide by zero encountered in divide
  label_ratio = neu_nonmaze.firing_rate/neu_slice.firing_rate
/var/folders/59/t44jf7m12zb1989pvvf2p6bm0000gn/T/ipykernel_1458/3303140680.py:15: RuntimeWarning: divide by zero encountered in divide
  label_ratio = neu_nonmaze.firing_rate/neu_slice.firing_rate


sess_RatS_Day2NSD


/var/folders/59/t44jf7m12zb1989pvvf2p6bm0000gn/T/ipykernel_1458/3303140680.py:15: RuntimeWarning: divide by zero encountered in divide
  label_ratio = neu_nonmaze.firing_rate/neu_slice.firing_rate
/var/folders/59/t44jf7m12zb1989pvvf2p6bm0000gn/T/ipykernel_1458/3303140680.py:15: RuntimeWarning: divide by zero encountered in divide
  label_ratio = neu_nonmaze.firing_rate/neu_slice.firing_rate
/var/folders/59/t44jf7m12zb1989pvvf2p6bm0000gn/T/ipykernel_1458/3303140680.py:15: RuntimeWarning: invalid value encountered in divide
  label_ratio = neu_nonmaze.firing_rate/neu_slice.firing_rate


sess_RatR_Day1NSD
sess_RatU_Day2NSD


...

sess_RatV_Day3NSD


/var/folders/59/t44jf7m12zb1989pvvf2p6bm0000gn/T/ipykernel_1458/3303140680.py:15: RuntimeWarning: divide by zero encountered in divide
  label_ratio = neu_nonmaze.firing_rate/neu_slice.firing_rate
/var/folders/59/t44jf7m12zb1989pvvf2p6bm0000gn/T/ipykernel_1458/3303140680.py:15: RuntimeWarning: divide by zero encountered in divide
  label_ratio = neu_nonmaze.firing_rate/neu_slice.firing_rate
/var/folders/59/t44jf7m12zb1989pvvf2p6bm0000gn/T/ipykernel_1458/3303140680.py:15: RuntimeWarning: divide by zero encountered in divide
  label_ratio = neu_nonmaze.firing_rate/neu_slice.firing_rate


sess_RatJ_Day1
sess_RatK_Day1
sess_RatN_Day1
sess_RatS_Day3SD
sess_RatR_Day2SD


/var/folders/59/t44jf7m12zb1989pvvf2p6bm0000gn/T/ipykernel_1458/3303140680.py:15: RuntimeWarning: divide by zero encountered in divide
  label_ratio = neu_nonmaze.firing_rate/neu_slice.firing_rate


KeyboardInterrupt: 

In [3]:
nd2.tag_neurons('firing_during_maze', label_firing_all,values=label_firing_all_values)
nd2.tag_neurons('maze_vs_non_frate_ratio', label_ratio_all,values=label_ratio_all_values)

In [4]:
nd2.save_tags(cd2.save_path)

['firing_during_maze', 'maze_vs_non_frate_ratio']

### stats tests backend

In [ ]:
import neuropy.ui.stats_tests_backend as st
b = st.StatsTestBackend(cd2,sd2)

In [9]:
b.load('../data/project_test2/stats_results/post2-nsd-best.json')

{'rows': [{'id': 63,
   'name': 'A',
   'color': '#8fb3ff',
   'sessions': ['RatJ_Day2',
    'RatK_Day2',
    'RatN_Day2',
    'RatS_Day2NSD',
    'RatR_Day1NSD',
    'RatU_Day2NSD',
    'RatV_Day1NSD',
    'RatV_Day3NSD'],
   'conn_types': ['E PYR→INT'],
   'segments': ['post_half1'],
   'resolution': ['lowres', 'highres'],
   'groups': ['best'],
   'data_type': 'Conn Strength',
   'follow': None,
   'follow_pickers': [],
   'follow_exclude': False},
  {'id': 64,
   'name': 'B',
   'color': '#ffb3b3',
   'sessions': ['RatJ_Day2',
    'RatK_Day2',
    'RatN_Day2',
    'RatS_Day2NSD',
    'RatR_Day1NSD',
    'RatU_Day2NSD',
    'RatV_Day1NSD',
    'RatV_Day3NSD'],
   'conn_types': ['E PYR→INT'],
   'segments': ['post_half2'],
   'resolution': ['lowres', 'highres'],
   'groups': ['best'],
   'data_type': 'Conn Strength',
   'follow': None,
   'follow_pickers': ['sess', 'ct', 'res', 'grp'],
   'follow_exclude': False}],
 'test': {'test_type': 'Pairwise t-test',
  'sides': 'Two-sided',
  '

In [ ]:
b.run_test(a_vals=post1,b_vals=post2,a_label='post1',b_label='post2',test_type=None,alpha=0.05,alternative='two-sided',nonparametric=False,log=False)

In [ ]:
import matplotlib.pyplot as plt
for key in nd2.session_keys():
    if key.session=='RatU_Day2NSD':
        print(nd2.get_themes(key)['paradigm'])
        print(key)
        neu=nd2.neurons_for(key)
        trains = neu.get_binned_spiketrains()
        for train in trains.spike_counts:
            plt.gca().set_aspect(3000) # reasonable aspect making sure plot is flat but not collapsed
            plt.plot(train)
            plt.show()

In [ ]:
np.argsort(neu.behav_slice(behav_times=themes,labels='maze').firing_rate)[::-1]

array([11,  6,  3,  0, 15, 13,  4,  5, 10, 14,  8, 16,  9,  7, 12,  2,  1])

1.79
0.61
0.45
0.36
0.34
0.17
0.07
0.05
0.01
0.01
0.00
0.00
0.00
0.00
0.00
0.00
0.00


In [26]:
import numpy as np
np.sort(neu.firing_rate)[::-1]

array([0.79260011, 0.59376166, 0.467628  , 0.41722996, 0.40493354,
       0.35070801, 0.21362881, 0.20955886, 0.17301595, 0.12225422,
       0.11537861, 0.1094209 , 0.0985273 , 0.07928599, 0.06951812,
       0.04483867, 0.03711443])

In [40]:
list(nd.session_keys())

[Key(session='RatJ_Day2', segment=None, excitability=None, conn_type=None, resolution='lowres', ref=None, tgt=None),
 Key(session='RatK_Day2', segment=None, excitability=None, conn_type=None, resolution='lowres', ref=None, tgt=None),
 Key(session='RatN_Day2', segment=None, excitability=None, conn_type=None, resolution='lowres', ref=None, tgt=None),
 Key(session='RatS_Day2NSD', segment=None, excitability=None, conn_type=None, resolution='lowres', ref=None, tgt=None),
 Key(session='RatR_Day1NSD', segment=None, excitability=None, conn_type=None, resolution='lowres', ref=None, tgt=None),
 Key(session='RatU_Day2NSD', segment=None, excitability=None, conn_type=None, resolution='lowres', ref=None, tgt=None),
 Key(session='RatV_Day1NSD', segment=None, excitability=None, conn_type=None, resolution='lowres', ref=None, tgt=None),
 Key(session='RatV_Day3NSD', segment=None, excitability=None, conn_type=None, resolution='lowres', ref=None, tgt=None),
 Key(session='RatJ_Day1', segment=None, excitabil

In [83]:
for s in nd._sessions:
 x,y,z=len(s.recinfo.skipped_channels), len(s.recinfo.discarded_channels), s.neurons.waveforms[0].shape[0]
 print((x+y+z)/16,x+y,z,x+y+z)

3.8125 7 54 61
8.375 10 124 134
7.875 10 116 126
11.9375 16 175 191
8.0 49 79 128
12.0 3 189 192
16.0 1 255 256
16.0 1 255 256
4.75 15 61 76
8.1875 70 61 131
8.1875 10 121 131
11.9375 18 173 191
8.0 49 79 128
12.0 3 189 192
12.0 3 189 192
16.0 1 255 256


In [ ]:
for x in nd.neurons_for(list(nd.session_keys())[3]).waveforms:
    print(x.argmax()//61)

56
56
95
96
96
136
141
168
49
63
89
110
105
137
141
153
58
58
78
111
142
3
75
91
95
107
123
142
141
155
157
157
96
126
156
53
125
141
58
126
126
142
156
156
55
55
95
111
111
151
151
53
108
137
48
60
89
123
60
94
138
124
139
123
142
140
42
168
57
15
3
138
62
157
111
154
79
63
138
80
95
23
154
60
110
123
123
64
62
110
134
59
62
4
110
74
96
95
126
111
91
46
96
153
106
125


In [29]:
import numpy as np, pandas as pd
from neuropy.analyses.ms_connectivity import Key, CCGSourceConfig
SEG = 'maze'
rows = []
for sk in nd.session_keys():
    key = Key(session=str(sk.session), segment=SEG)
    src = CCGSourceConfig(key=key)
    src._root = cd.save_path
    try:
        src.load()
    except FileNotFoundError:
        continue
    n = nd.neurons_for(key.nd())
    for i, r in enumerate(src.firing_rates):
        rows.append(dict(session=str(sk.session), neuron=int(n.neuron_ids[i]),
                         ntype=n.neuron_type[i], fr=r,
                         active_h=src.total_time_hours))
df = pd.DataFrame(rows)

In [30]:
df

,session,neuron,ntype,fr,active_h
0,RatJ_Day2,7,pyr,0.362760,1.066667
1,RatJ_Day2,9,pyr,0.000000,1.066667
2,RatJ_Day2,10,pyr,0.000000,1.066667
3,RatJ_Day2,11,pyr,0.448177,1.066667
4,RatJ_Day2,12,pyr,0.071354,1.066667
...,...,...,...,...,...
1280,RatV_Day2SD,96,inter,16.258730,0.875000
1281,RatV_Day2SD,97,pyr,0.244127,0.875000
1282,RatV_Day2SD,98,inter,0.347302,0.875000
1283,RatV_Day2SD,100,inter,0.146032,0.875000


In [37]:
import pynwb, numpy as np, glob
for p in sorted(glob.glob('/Users/selinl/Documents/ms_synchrony/dandi001695/*/*.nwb')):
    with pynwb.NWBHDF5IO(p,'r') as io:
        f=io.read()
        print('='*70); print(p.split('/')[-1])
        print(' desc     :', (f.session_description or '')[:150])
        print(' exp_desc :', (f.experiment_description or '')[:200])
        print(' subject  :', f.subject.subject_id if f.subject else None,
              '| species:', getattr(f.subject,'species',None) if f.subject else None)
        print(' start    :', f.session_start_time)
        u=f.units.to_dataframe()
        print(' units    :', len(u), '| areas:', sorted(set(u.cell_area.astype(str))))
        print(' keywords :', list(f.keywords[:]) if f.keywords is not None else None)

sub-M01_ses-20240308T100000_ecephys.nwb
 desc     : Head Fix Session with Odor Stimulation
 exp_desc : 
 subject  : M01 | species: Mus musculus
 start    : 2024-03-08 10:00:00+00:00
 units    : 541 | areas: ['CA1', 'CA3', 'RSC']
 keywords : None
sub-M01_ses-20240312T100000_behavior+ecephys.nwb
 desc     : Novel Maze Session with Pre and Post sleep
 exp_desc : 
 subject  : M01 | species: Mus musculus
 start    : 2024-03-12 10:00:00+00:00
 units    : 430 | areas: ['CA1', 'CA3', 'RSC']
 keywords : None
sub-M01_ses-20240313T100000_behavior+ecephys.nwb
 desc     : Novel Maze Session with Pre and Post sleep
 exp_desc : 
 subject  : M01 | species: Mus musculus
 start    : 2024-03-13 10:00:00+00:00
 units    : 240 | areas: ['CA1', 'CA3', 'RSC']
 keywords : None
sub-M01_ses-20240314T100000_behavior+ecephys.nwb
 desc     : Novel Maze Session with Pre and Post sleep
 exp_desc : 
 subject  : M01 | species: Mus musculus
 start    : 2024-03-14 10:00:00+00:00
 units    : 222 | areas: ['CA1', 'CA3', '

In [131]:
import pynwb, numpy as np, glob
for p in sorted(glob.glob('/Users/selinl/Documents/ms_synchrony/dandi001695/*/*.nwb')):
    with pynwb.NWBHDF5IO(p,'r') as io:
        f=io.read()
        if f.electrodes is not None:
            print(f.electrodes.to_dataframe())

    location                                              group  \
id                                                                
0        CA1  ElectrodeGroup1 pynwb.ecephys.ElectrodeGroup a...   
1        CA1  ElectrodeGroup1 pynwb.ecephys.ElectrodeGroup a...   
2        CA1  ElectrodeGroup1 pynwb.ecephys.ElectrodeGroup a...   
3        CA1  ElectrodeGroup1 pynwb.ecephys.ElectrodeGroup a...   
4        CA1  ElectrodeGroup1 pynwb.ecephys.ElectrodeGroup a...   
..       ...                                                ...   
379      CA1  ElectrodeGroup1 pynwb.ecephys.ElectrodeGroup a...   
380      CA1  ElectrodeGroup1 pynwb.ecephys.ElectrodeGroup a...   
381      CA1  ElectrodeGroup1 pynwb.ecephys.ElectrodeGroup a...   
382      CA1  ElectrodeGroup1 pynwb.ecephys.ElectrodeGroup a...   
383      CA1  ElectrodeGroup1 pynwb.ecephys.ElectrodeGroup a...   

          group_name   x   y   z  imp        filtering  
id                                                      
0    Electrode

In [ ]:
from neuropy.core.nwb_session import NWBDataset
from neuropy.io.fieldmap import FieldMap
from neuropy.io.nwbio import UNITS_SCHEMA, DANDI_001695

ds = NWBDataset('/Users/selinl/Documents/ms_synchrony/001695')
for sess in ds.sessions(FieldMap(UNITS_SCHEMA, DANDI_001695)):
    for name, ep in sess.themes.items():
        print(sess._session_name, name, ep.get_unique_labels(), ep.n_epochs)

sub-M01_ses-20240308T100000_ecephys Odor Stimulus ['Odor 1' 'Odor Empty'] 401
sub-M01_ses-20240312T100000_behavior+ecephys SleepStates ['NREM' 'REM' 'Ripple' 'WAKE'] 145
sub-M01_ses-20240313T100000_behavior+ecephys SleepStates ['NREM' 'REM' 'Ripple' 'WAKE'] 1863
sub-M01_ses-20240314T100000_behavior+ecephys SleepStates ['NREM' 'REM' 'Ripple' 'WAKE'] 1132
sub-M01_ses-20240318T100000_behavior+ecephys SleepStates ['NREM' 'REM' 'Ripple' 'WAKE'] 463
sub-M02_ses-20240226T100000_ecephys Odor Stimulus ['Odor 1' 'Odor Empty'] 81
sub-M02_ses-20240307T100000_ecephys Odor Stimulus ['Odor 1' 'Odor Empty'] 401
sub-M02_ses-20240312T100000_behavior+ecephys SleepStates ['NREM' 'REM' 'Ripple' 'WAKE'] 2846
sub-M02_ses-20240313T100000_behavior+ecephys SleepStates ['NREM' 'REM' 'Ripple' 'WAKE'] 3463
sub-M02_ses-20240314T100000_behavior+ecephys SleepStates ['NREM' 'REM' 'Ripple' 'WAKE'] 3278
sub-M02_ses-20240318T100000_behavior+ecephys SleepStates ['NREM' 'REM' 'Ripple' 'WAKE'] 2068
sub-M03_ses-20240621T1000

In [16]:
list(s.__dict__.keys())

['path', 'fields_by_file']

In [17]:
s._session_name

AttributeError: 'NWBDataset' object has no attribute '_session_name'

In [ ]:
f = s._nwb._nwb
list(f.units.colnames)

AttributeError: 'NWBDataset' object has no attribute '_nwb'

In [ ]:
from neuropy.core.nwb_session import NWBSession
sess = [NWBSession("/Users/selinl/Documents/ms_synchrony/001695/sub-M01/sub-M01_ses-20240312T100000_behavior+ecephys.nwb", epoch_table='epochs', neuron_type_col='cell_type')]

In [ ]:
from neuropy.productivity.prompt_space import PromptSpace
PromptSpace()

```
source ~/miniconda/bin/activate
conda activate NeuroPy2
-----updated
source ~/miniforge3/etc/profile.d/conda.sh
mamba activate NeuroPy2

/usr/local/MATLAB/R2025a/bin/matlab
```